# VBZ Geo-Map — Folium

Interaktive Kartenvisualisierung mit **Folium / Leaflet.js**.
Output ist eine HTML-Datei — kein Python zum Betrachten nötig.

| Karte | Inhalt |
| :---- | :----- |
| 1 | Stadtkreise + Tramlinien (VBZ-Farben) + Haltestellen |
| 2 | Verspätungs-Heatmap (HeatMap-Plugin) |
| 3 | Verspätung auf Streckenabschnitten (farbige PolyLines) |

> **Hinweis:** Verspätungsdaten sind synthetisch (Platzhalter bis Ist-Daten verfügbar).

Installation: `pip install folium`


---

## Architektur-Notizen — Folium

### Stärken
- **Segment-Einfärbung:** Jedes `PolyLine`-Element ist ein separates HTML-Objekt.
  Der Browser verarbeitet 200–400 Segmente problemlos.
- **Kein Python zum Öffnen:** HTML-Datei direkt im Browser.
- **HeatMap-Plugin:** Schnell und einfach für Dichte-Darstellungen.

### Einschränkungen
- **Skalierungsgrenze:** Ab ~1000 PolyLine-Elementen wird die HTML-Datei träge.
  Bei allen Tramlinien + allen Segmenten (~500) bleibt es knapp im grünen Bereich.
- **Kein VS Code Rendering:** Nur im Browser oder als gespeicherte HTML-Datei.
- **Kein Dashboard:** Nicht direkt in Dash oder Streamlit integrierbar.

### Empfehlung fürs Projekt
> Folium für **Weitergabe ohne Python** und **schnelle Prototypen**.
> Segment-Einfärbung für alle Linien gleichzeitig möglich, aber HTML-Datei wird groß (~5 MB).
> Für Gesamt-Netz-Übersicht lieber Kepler.gl.


In [11]:
import pandas as pd
from pathlib import Path

# Repo-Root finden, egal von wo Jupyter gestartet wurde
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "data" / "interim").exists():
        _ROOT = _p
        break

GTFS_DIR = _ROOT / "data" / "interim" / "vbz" / "gtfs"
GEO_DIR  = _ROOT / "data" / "raw" / "vbz" / "stadtkreise" / "data"

import json
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
from scipy.spatial import cKDTree


# Funktioniert egal ob cwd = notebook-ordner oder repo-root


# Repo-Root unabhängig vom Jupyter-Working-Directory ermitteln


routes = pd.read_parquet(GTFS_DIR / "gtfs_tram_routes.parquet")
stops  = pd.read_parquet(GTFS_DIR / "gtfs_tram_stops.parquet")
shapes = pd.read_parquet(GTFS_DIR / "gtfs_tram_shapes.parquet")
trips  = pd.read_parquet(GTFS_DIR / "gtfs_tram_trips.parquet")

with open(GEO_DIR / "stzh_adm_stadtkreise_v.json") as f:
    stadtkreise_geo = json.load(f)
with open(GEO_DIR / "stzh_adm_stadtkreise_beschr_p.json") as f:
    labels_geo = json.load(f)

# Referenzjahr 2024
routes_2024  = routes[routes["year"] == "2024"].copy()
shapes_2024  = shapes[shapes["year"] == "2024"].copy()
stops_2024   = stops[stops["year"] == "2024"].copy()
trips_2024   = trips[trips["year"] == "2024"].copy()
stops_unique = stops_2024.drop_duplicates(subset=["stop_name"]).copy()

# Synthetische Verspätung
np.random.seed(42)
stops_unique["delay_min"] = np.random.exponential(scale=1.5, size=len(stops_unique)).round(1)

# Farben-Mapping
line_colors = dict(zip(
    routes_2024["route_short_name"],
    "#" + routes_2024["route_color"]
))
kreis_map = {
    f["properties"]["objid"]: f["properties"]["kname"]
    for f in stadtkreise_geo["features"]
}

def best_shape_for_line(line_name):
    route_ids = routes_2024[routes_2024["route_short_name"] == line_name]["route_id"]
    shape_ids = trips_2024[trips_2024["route_id"].isin(route_ids)]["shape_id"].unique()
    if len(shape_ids) == 0:
        return None
    return max(shape_ids, key=lambda s: len(shapes_2024[shapes_2024["shape_id"] == s]))


def delay_to_hex(delay_min, max_delay=5.0):
    t = min(delay_min / max_delay, 1.0)
    r = int(255 * t)
    g = int(255 * (1 - t))
    return f"#{r:02x}{g:02x}00"


CENTER = [47.378, 8.540]
print(f"Stops: {len(stops_unique)}  Linien: {len(routes_2024)}")

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'reports').exists():
        REPORTS_DIR = _p / 'reports'
        break
REPORTS_DIR.mkdir(exist_ok=True)


Stops: 1189  Linien: 16


---

## Karte 1 — Stadtkreise + Tramnetz + Haltestellen


In [12]:
m1 = folium.Map(
    location=CENTER,
    zoom_start=13,
    tiles="CartoDB dark_matter"
)

# ── Stadtkreis-Polygone ───────────────────────────────────────────────────
folium.GeoJson(
    stadtkreise_geo,
    name="Stadtkreise",
    style_function=lambda _: {
        "fillColor": "#3a3a5e",
        "color":     "#aaaacc",
        "weight":    1,
        "fillOpacity": 0.15
    },
    tooltip=folium.GeoJsonTooltip(fields=["kname"], aliases=["Stadtkreis:"])
).add_to(m1)

# ── Stadtkreis-Labels ─────────────────────────────────────────────────────
for feature in labels_geo["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    name = kreis_map.get(feature["properties"]["objid"], "?")
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;color:rgba(255,255,255,0.6);font-weight:bold">'
                 f'{name}</div>',
            icon_size=(60, 20),
            icon_anchor=(30, 10)
        )
    ).add_to(m1)

# ── Tramlinien ────────────────────────────────────────────────────────────
for line in sorted(routes_2024["route_short_name"].unique()):
    color    = line_colors.get(line, "#999999")
    shape_id = best_shape_for_line(line)
    if shape_id is None:
        continue
    pts = (
        shapes_2024[shapes_2024["shape_id"] == shape_id]
        .sort_values("shape_pt_sequence")
        .iloc[::3]
    )
    coords = list(zip(pts["shape_pt_lat"], pts["shape_pt_lon"]))
    folium.PolyLine(
        locations=coords,
        color=color,
        weight=2.5,
        opacity=0.9,
        tooltip=f"Linie {line}"
    ).add_to(m1)

# ── Haltestellen ──────────────────────────────────────────────────────────
stop_layer = folium.FeatureGroup(name="Haltestellen")
for _, row in stops_unique.iterrows():
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=3,
        color="white",
        fill=True,
        fill_opacity=0.75,
        tooltip=row["stop_name"]
    ).add_to(stop_layer)
stop_layer.add_to(m1)

folium.LayerControl().add_to(m1)

m1.save(REPORTS_DIR / "folium_tramnetz.html")
print("Gespeichert: vbz_tramnetz_folium.html")
m1

Gespeichert: vbz_tramnetz_folium.html


---

## Karte 2 — Verspätungs-Heatmap

Foliums `HeatMap`-Plugin visualisiert Dichte gewichtet nach Verspätung.


In [13]:
m2 = folium.Map(
    location=CENTER,
    zoom_start=13,
    tiles="CartoDB dark_matter"
)

# Stadtkreise
folium.GeoJson(
    stadtkreise_geo,
    style_function=lambda _: {
        "fillColor": "transparent",
        "color":     "#aaaacc",
        "weight":    1,
        "fillOpacity": 0
    },
    tooltip=folium.GeoJsonTooltip(fields=["kname"], aliases=["Stadtkreis:"])
).add_to(m2)

# Heatmap gewichtet nach Verspätung
heat_data = [
    [row["stop_lat"], row["stop_lon"], row["delay_min"]]
    for _, row in stops_unique.iterrows()
]
HeatMap(
    heat_data,
    name="Verspätungs-Heatmap",
    min_opacity=0.3,
    radius=20,
    blur=15,
    gradient={0.0: "blue", 0.4: "lime", 0.65: "yellow", 1.0: "red"}
).add_to(m2)

folium.LayerControl().add_to(m2)

m2.save(REPORTS_DIR / "folium_heatmap.html")
print("Gespeichert: vbz_heatmap_folium.html")
m2

Gespeichert: vbz_heatmap_folium.html


### Karte 2b — Haltestellen eingefärbt nach Verspätung


In [14]:
m2b = folium.Map(
    location=CENTER,
    zoom_start=13,
    tiles="CartoDB dark_matter"
)

folium.GeoJson(
    stadtkreise_geo,
    style_function=lambda _: {
        "fillColor": "#2a2a4e",
        "color":     "#aaaacc",
        "weight":    1,
        "fillOpacity": 0.1
    }
).add_to(m2b)

max_delay = stops_unique["delay_min"].quantile(0.95)

for _, row in stops_unique.iterrows():
    color = delay_to_hex(row["delay_min"], max_delay)
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=f"{row['stop_name']}: {row['delay_min']:.1f} Min"
    ).add_to(m2b)

m2b.save(REPORTS_DIR / "folium_stops_delay.html")
print("Gespeichert: vbz_stops_delay_folium.html")
m2b

Gespeichert: vbz_stops_delay_folium.html


---

## Karte 3 — Verspätung auf Streckenabschnitten

Jeder Streckenabschnitt wird als farbige `PolyLine` dargestellt.
Tooltip zeigt Haltestellenpaar und Verspätung.


In [15]:
m3 = folium.Map(
    location=CENTER,
    zoom_start=13,
    tiles="CartoDB dark_matter"
)

# Stadtkreise
folium.GeoJson(
    stadtkreise_geo,
    style_function=lambda _: {
        "fillColor": "transparent",
        "color":     "#aaaacc",
        "weight":    1,
        "fillOpacity": 0
    }
).add_to(m3)

max_delay = stops_unique["delay_min"].quantile(0.95)

for line in sorted(routes_2024["route_short_name"].unique()):
    shape_id = best_shape_for_line(line)
    if shape_id is None:
        continue

    shape_pts = (
        shapes_2024[shapes_2024["shape_id"] == shape_id]
        .sort_values("shape_pt_sequence")
        .reset_index(drop=True)
    )

    tree = cKDTree(shape_pts[["shape_pt_lat", "shape_pt_lon"]].values)

    lat_min = shape_pts["shape_pt_lat"].min() - 0.005
    lat_max = shape_pts["shape_pt_lat"].max() + 0.005
    lon_min = shape_pts["shape_pt_lon"].min() - 0.005
    lon_max = shape_pts["shape_pt_lon"].max() + 0.005

    nearby = stops_unique[
        stops_unique["stop_lat"].between(lat_min, lat_max) &
        stops_unique["stop_lon"].between(lon_min, lon_max)
    ].copy()

    if len(nearby) < 2:
        continue

    _, idxs = tree.query(nearby[["stop_lat", "stop_lon"]].values)
    nearby["shape_idx"] = idxs
    nearby = nearby.sort_values("shape_idx").reset_index(drop=True)

    for i in range(len(nearby) - 1):
        s1 = nearby.iloc[i]
        s2 = nearby.iloc[i + 1]
        seg_delay = (s1["delay_min"] + s2["delay_min"]) / 2
        color     = delay_to_hex(seg_delay, max_delay)

        lo = int(min(s1["shape_idx"], s2["shape_idx"]))
        hi = int(max(s1["shape_idx"], s2["shape_idx"]))
        seg = shape_pts.iloc[lo:hi + 1]

        if len(seg) < 2:
            continue

        coords = list(zip(seg["shape_pt_lat"], seg["shape_pt_lon"]))
        folium.PolyLine(
            locations=coords,
            color=color,
            weight=4,
            opacity=0.85,
            tooltip=f"Linie {line}: {s1['stop_name']} → {s2['stop_name']} ({seg_delay:.1f} Min)"
        ).add_to(m3)

# Haltestellen
for _, row in stops_unique.iterrows():
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=3,
        color="white",
        fill=True,
        fill_opacity=0.6,
        tooltip=f"{row['stop_name']}: {row['delay_min']:.1f} Min"
    ).add_to(m3)

m3.save(REPORTS_DIR / "folium_strecken_linie11.html")
print("Gespeichert: vbz_strecken_folium.html")
m3

Gespeichert: vbz_strecken_folium.html


In [16]:
# Karte 3: Linie 11 — zwei Segmente mit Verspätung (Beispiel, identisch mit Plotly/GeoPandas)

stops_2024_local = stops[stops["year"] == "2024"].copy()

h1 = stops_2024_local[stops_2024_local["stop_name"] == "Zürich, Paradeplatz"].iloc[0]
h2 = stops_2024_local[stops_2024_local["stop_name"] == "Zürich, Bellevue"].iloc[0]
h3 = stops_2024_local[stops_2024_local["stop_name"] == "Zürich, Bahnhof Stadelhofen"].iloc[0]

route_ids_11 = routes_2024[routes_2024["route_short_name"] == "11"]["route_id"]
shape_ids_11 = trips_2024[trips_2024["route_id"].isin(route_ids_11)]["shape_id"].unique()
shape_id_11  = max(shape_ids_11, key=lambda s: len(shapes_2024[shapes_2024["shape_id"] == s]))
shape_11     = shapes_2024[shapes_2024["shape_id"] == shape_id_11].sort_values("shape_pt_sequence")
color_11     = "#" + routes_2024[routes_2024["route_short_name"] == "11"]["route_color"].iloc[0]

m3 = folium.Map(location=[47.3695, 8.5430], zoom_start=14, tiles="CartoDB dark_matter")

# Stadtkreise
folium.GeoJson(stadtkreise_geo, style_function=lambda _: {
    "fillColor": "transparent", "color": "#aaaacc", "weight": 1, "fillOpacity": 0
}).add_to(m3)

# Linie 11 in VBZ-Farbe
coords_11 = list(zip(shape_11["shape_pt_lat"], shape_11["shape_pt_lon"]))
folium.PolyLine(coords_11, color=color_11, weight=3, opacity=0.9, tooltip="Linie 11").add_to(m3)

# Segment gelb: Paradeplatz -> Bellevue
folium.PolyLine(
    [[h1["stop_lat"], h1["stop_lon"]], [h2["stop_lat"], h2["stop_lon"]]],
    color="#FFD700", weight=8, opacity=0.9,
    tooltip="Paradeplatz → Bellevue: 2.5 Min"
).add_to(m3)

# Segment rot: Bellevue -> Stadelhofen
folium.PolyLine(
    [[h2["stop_lat"], h2["stop_lon"]], [h3["stop_lat"], h3["stop_lon"]]],
    color="#E53935", weight=8, opacity=0.9,
    tooltip="Bellevue → Stadelhofen: 5.1 Min"
).add_to(m3)

# Haltestellen
for h, name, delay in [(h1, "Paradeplatz", 2.5), (h2, "Bellevue", 3.8), (h3, "Stadelhofen", 5.1)]:
    folium.CircleMarker(
        location=[h["stop_lat"], h["stop_lon"]],
        radius=8, color="white", fill=True, fill_opacity=0.9,
        tooltip=f"{name}: {delay} Min"
    ).add_to(m3)

m3.save(REPORTS_DIR / "folium_strecken_linie11.html")
print("Gespeichert: vbz_strecken_folium.html")
m3


Gespeichert: vbz_strecken_folium.html
